In [3]:
import shutil
from pathlib import Path

import lancedb
from lancedb.rerankers import ColbertReranker

from dreamai.ai import ModelName
from dreamai.lance_utils import get_user_query
from dreamai.search_actions import add_data_with_descriptions
from dreamai.settings import CreatorSettings, RAGAppSettings, RAGSettings

creator_settings = CreatorSettings()
rag_settings = RAGSettings()
rag_app_settings = RAGAppSettings()

MODEL = creator_settings.model
LANCE_URI = rag_settings.lance_uri
RERANKER = rag_settings.reranker
HAS_WEB = rag_app_settings.has_web
DATA = "/media/hamza/data2/RFP/docs"

lance_db = lancedb.connect(uri="burr_app/lance/rag/")
reranker = ColbertReranker(RERANKER)

%load_ext autoreload
%autoreload 2
%reload_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
lance_db.table_names()

['ApplicationSecurityStandards',
 'BusinessContinuityDisasterRecovery',
 'CompanySecurityPolicy',
 'EmployeeSecurityPolicy',
 'VendorManagement']

In [6]:
table = lance_db.open_table("VendorManagement")
table.to_pandas()

,name,index,text,vector
0,vendor-management-policy.md,0,# Acme Inc Vendor Management Policy ## Governa...,"[-0.0021941871, -0.018549044, 0.023975216, 0.0..."
1,vendor-management-policy.md,1,c . - Vendors are screened through public reco...,"[-0.011064428, -0.037052516, 0.0054052384, 0.0..."
2,vendor-management-policy.md,2,iewed annually . ## Contracting - Contracts in...,"[0.008778638, -0.005554448, 0.01256999, 0.0427..."
3,vendor-management-policy.md,3,critical solutions . ## Onboarding - Upon cont...,"[-0.0056179445, -0.013572615, 0.008454074, 0.0..."
4,vendor-management-policy.md,4,history are reviewed for increased risk . - V...,"[0.008876327, -0.021689912, 0.013347239, 0.040..."


In [7]:
query = "what is the vendor management system?"
table.search(query=query, query_type="hybrid").rerank(reranker=reranker).limit(10).to_pandas()

/home/hamza/dev/dreamai/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/home/hamza/dev/dreamai/.venv/lib/python3.12/site-packages/sentence_transformers/models/Dense.py:89: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unp

,name,index,vector,text,_relevance_score
0,vendor-management-policy.md,0,"[-0.0021941871, -0.018549044, 0.023975216, 0.0...",# Acme Inc Vendor Management Policy ## Governa...,0.924227
1,vendor-management-policy.md,3,"[-0.0056179445, -0.013572615, 0.008454074, 0.0...",critical solutions . ## Onboarding - Upon cont...,0.908225
2,vendor-management-policy.md,1,"[-0.011064428, -0.037052516, 0.0054052384, 0.0...",c . - Vendors are screened through public reco...,0.906300
3,vendor-management-policy.md,4,"[0.008876327, -0.021689912, 0.013347239, 0.040...",history are reviewed for increased risk . - V...,0.892939
4,vendor-management-policy.md,2,"[0.008778638, -0.005554448, 0.01256999, 0.0427...",iewed annually . ## Contracting - Contracts in...,0.875413


In [ ]:
table.search(where="vendor_management_system = 'yes'").to_pandas()

In [5]:
from pydantic import BaseModel, Field, ValidationInfo
from typing import Literal, Self
import sys
from pathlib import Path


# Import the necessary classes and constants
from dreamai.dialog_models import SourcedSentence, SourcedResponse, MAX_RESPONSE_SENTENCES

# Sample data
documents = [
    {"name": "Document 1", "index": 10},
    {"name": "Document 2", "index": 11},
    {"name": "Document 3", "index": 12},
]
context = {"documents": documents, "max_non_sourced_factor": 0.5}

# Create some SourcedSentence instances
sentence1 = SourcedSentence(text="This is the first sentence.", sources=[0, 1])
sentence2 = SourcedSentence(text="This is the second sentence.", sources=[1, 2])
sentence3 = SourcedSentence(text="This is an unsourced sentence.", sources=[-1])

# Create a SourcedResponse
response = SourcedResponse(sentences=[sentence1, sentence2, sentence3])

# Validate the response
validated_response = response.model_validate(response.model_dump(), context=context)

# Print the string representation of the response
print(str(validated_response))

# Print some additional information
print("\nAdditional Information:")
print(f"Is the response sure? {validated_response._sure}")
print(f"Source documents: {validated_response._source_docs}")

# Test a single-sentence response
single_sentence_response = SourcedResponse(sentences=[sentence1])
print("\nSingle Sentence Response:")
print(str(single_sentence_response))

- This is the first sentence. [0, 1]
- This is the second sentence. [1, 2]
- This is an unsourced sentence. [-1]

Additional Information:
Is the response sure? True
Source documents: [[{'name': 'Document 1', 'index': 10}, {'name': 'Document 2', 'index': 11}], [{'name': 'Document 2', 'index': 11}, {'name': 'Document 3', 'index': 12}], []]

Single Sentence Response:
This is the first sentence. [0, 1]


In [12]:
single_sentence_response._sure

True

In [23]:
src = SourcedResponse(
    sentences=[
        SourcedSentence(text="hello", sources=[2]),
        SourcedSentence(text="hello2", sources=[-1]),
        SourcedSentence(text="hello3", sources=[-1]),
    ]
)

In [6]:
print('- Acme Inc has a comprehensive Business Continuity and Disaster Recovery Plan that outlines processes for responding to various disruptions. [0, 1, 2, 3, 4]\n- The plan addresses recovery time objectives (RTOs), recovery sequence, roles and responsibilities, notification procedures, alternate site locations, and detailed recovery procedures. [0, 1, 2, 3, 4]\n- It covers scenarios such as loss of facility, technology, suppliers, and people (pandemic). [0, 1, 2, 3, 4]\n- The plan also includes call trees for crisis management, recovery teams, employees, customers, and public authorities. [0, 1, 2, 3, 4]\n- Failover procedures are documented for network rerouting, DNS updates, and other critical functions. [0, 1, 2, 3, 4]\n- The plan is regularly exercised through walkthroughs, functional exercises, and full interruption tests to ensure its effectiveness. [3, 4]\n- After-action reviews are conducted to document lessons learned and update the plan. [3, 4]\n- The plan is continuously improved by reevaluating recovery capabilities after significant business or technology changes, reviewing supply chain resiliency annually, and monitoring threat intelligence services. [1, 2, 4]\n- The BC/DR program is sponsored by the COO and managed by the BC/DR Manager with support from various departments. [2]\n- A Business Impact Analysis (BIA) is conducted annually to identify critical processes, resources, dependencies, and financial/operational impacts of a disruption. [2]')

- Acme Inc has a comprehensive Business Continuity and Disaster Recovery Plan that outlines processes for responding to various disruptions. [0, 1, 2, 3, 4]
- The plan addresses recovery time objectives (RTOs), recovery sequence, roles and responsibilities, notification procedures, alternate site locations, and detailed recovery procedures. [0, 1, 2, 3, 4]
- It covers scenarios such as loss of facility, technology, suppliers, and people (pandemic). [0, 1, 2, 3, 4]
- The plan also includes call trees for crisis management, recovery teams, employees, customers, and public authorities. [0, 1, 2, 3, 4]
- Failover procedures are documented for network rerouting, DNS updates, and other critical functions. [0, 1, 2, 3, 4]
- The plan is regularly exercised through walkthroughs, functional exercises, and full interruption tests to ensure its effectiveness. [3, 4]
- After-action reviews are conducted to document lessons learned and update the plan. [3, 4]
- The plan is continuously improved by r

In [25]:
src._sure

False

In [2]:
md = MarkdownData.model_validate(
    {"name": "test", "markdown": "lets' gooooooooooooooooooooo"},
    context={"chunk_size": 0, "separators": [r" "]},
)

In [8]:
class ModelA(BaseModel):
    a: int = Field(serialization_alias="a_alias")
    b: str


class ModelB(BaseModel):
    a: int
    b: str

In [9]:
a = ModelA(a=1, b="b")
a.model_dump(by_alias=True)

ValidationError: 1 validation error for ModelA
a_alias
  Field required [type=missing, input_value={'a': 1, 'b': 'b'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.7/v/missing

In [2]:
folder = Path("/media/hamza/data2/RFP/")
qs_md = data_to_md(folder / "questions", chunk_size=0)

In [3]:
docs_md = data_to_md("docs.md", chunk_size=0)

In [8]:
docs_md[0]

MarkdownResult(name='docs.md', markdown='# Acme Inc IT Security Policy\n\n## Access Control\n- All users are assigned unique user IDs and passwords. Sharing of accounts is strictly prohibited. \n- Passwords must be at least 12 characters long and include uppercase, lowercase, numbers and symbols. Passwords expire every 90 days. Previous 12 passwords cannot be reused.\n- Multi-factor authentication (MFA) is required for all remote access to company systems, cloud services, and privileged accounts.\n- User access is granted based on least privilege and job responsibilities. Access is reviewed quarterly and immediately revoked upon role change or termination.\n- Privileged access requires management approval and use of a separate admin account. Admin activities are logged and monitored.\n- Third-party access requires management approval and must use MFA. Access is disabled when no longer needed.\n\n## Data Protection \n- Customer data is classified as confidential and access restricted to

In [5]:
Path("docs.md").write_text("\n\n".join([d.markdown for d in docs_md]))

19363

In [ ]:
If it is asked to assist with tasks involving the expression of views held by a significant number of people, Claude provides assistance with the task regardless of its own views. If asked about controversial topics, it tries to provide careful thoughts and clear information. It presents the requested information without explicitly saying that the topic is sensitive, and without claiming to be presenting objective facts. When presented with a math problem, logic problem, or other problem benefiting from systematic thinking, Claude thinks through it step by step before giving its final answer. If Claude cannot or will not perform a task, it tells the user this without apologizing to them. It avoids starting its responses with “I’m sorry” or “I apologize”. If Claude is asked about a very obscure person, object, or topic, i.e. if it is asked for the kind of information that is unlikely to be found more than once or twice on the internet, Claude ends its response by reminding the user that although it tries to be accurate, it may hallucinate in response to questions like this. It uses the term ‘hallucinate’ to describe this since the user will understand what it means. If Claude mentions or cites particular articles, papers, or books, it always lets the human know that it doesn’t have access to search or a database and may hallucinate citations, so the human should double check its citations. Claude is very smart and intellectually curious. It enjoys hearing what humans think on an issue and engaging in discussion on a wide variety of topics. Claude uses markdown for code. Immediately after closing coding markdown, Claude asks the user if they would like it to explain or break down the code. It does not explain or break down the code unless the user explicitly requests it.Claude provides thorough responses to more complex and open-ended questions or to anything where a long response is requested, but concise responses to simpler questions and tasks. All else being equal, it tries to give the most correct and concise answer it can to the user’s message. Rather than giving a long response, it gives a concise response and offers to elaborate if further inforpmation may be helpful.

Claude is happy to help with analysis, question answering, math, coding, creative writing, teaching, role-play, general discussion, and all sorts of other tasks.

Claude responds directly to all human messages without unnecessary affirmations or filler phrases like “Certainly!”, “Of course!”, “Absolutely!”, “Great!”, “Sure!”, etc. Specifically, Claude avoids starting responses with the word “Certainly” in any way.